# Tutorial: Bandit Basics: UCB and Thompson Sampling

Learning goals:
- Understand the intuition behind Upper Confidence Bound (UCB) and Thompson Sampling.
- Read the core math without getting lost in notation.
- Work through a small example for each method.
- Run simple simulations and compare their behavior.


## Outline

1. Quick recap: what a bandit problem is
2. Why we need better exploration strategies
3. UCB intuition and math
4. UCB worked example
5. Thompson Sampling intuition and math
6. Thompson Sampling worked example
7. Simulating both methods on the same toy problem
8. Exercises and pitfalls


## Quick Recap: Multi-Armed Bandits

A multi-armed bandit problem is the simplest reinforcement learning setting. At each step, the agent chooses one action, gets one reward, and repeats.

There is no changing state here. The challenge is only this: **which arm should we pull next?**

This creates the classic tension:

- **exploration**: try arms to learn how good they are
- **exploitation**: use the arm that currently looks best


## Why Not Use Only Greedy Choice?

If we always pick the arm with the highest observed average reward, we can get stuck too early. A lucky arm may look best after only a few pulls, even if it is not truly the best arm.

Methods like UCB and Thompson Sampling try to balance caution and optimism in a more principled way than plain greedy choice.


In [1]:
from __future__ import annotations

import math
import random

random.seed(11)


## UCB Intuition

UCB stands for **Upper Confidence Bound**.

The core idea is optimistic: for each arm, do not look only at its current average reward. Also add an **uncertainty bonus**. If an arm has been pulled only a few times, we are less certain about it, so its bonus should be larger.

This makes sense:

- arms with high observed reward should look attractive
- arms with little data should also stay attractive for a while because they might still turn out to be very good


### UCB Math

A common UCB score for arm `i` at time `t` is:

`UCB_i = average_reward_i + sqrt((2 * log(t)) / n_i)`

where:

- `average_reward_i` is the sample mean reward of arm `i`
- `t` is the total number of pulls so far
- `n_i` is the number of times arm `i` has been pulled

Interpretation:

- the first term rewards arms that have done well
- the second term rewards arms we are uncertain about
- as `n_i` grows, the bonus shrinks because we have more evidence


## Worked UCB Example

Suppose after `t = 10` total pulls we have:

- arm A: average reward `0.60`, pulled `5` times
- arm B: average reward `0.50`, pulled `2` times
- arm C: average reward `0.40`, pulled `3` times

Even though arm A has the largest average reward, arm B is much less explored. UCB may still choose B because its uncertainty bonus is larger.


In [2]:
t = 10
ucb_summary = {
    'A': {'mean': 0.60, 'count': 5},
    'B': {'mean': 0.50, 'count': 2},
    'C': {'mean': 0.40, 'count': 3},
}

ucb_scores = {}
for arm, info in ucb_summary.items():
    bonus = math.sqrt((2 * math.log(t)) / info['count'])
    score = info['mean'] + bonus
    ucb_scores[arm] = {
        'mean': round(info['mean'], 4),
        'bonus': round(bonus, 4),
        'ucb_score': round(score, 4),
    }

ucb_scores


{'A': {'mean': 0.6, 'bonus': 0.9597, 'ucb_score': 1.5597},
 'B': {'mean': 0.5, 'bonus': 1.5174, 'ucb_score': 2.0174},
 'C': {'mean': 0.4, 'bonus': 1.239, 'ucb_score': 1.639}}

The worked example shows the logic directly. Arm B may win even with a lower sample mean, because the algorithm says: "we have not tried this one enough yet, so give it a chance."

That is the heart of UCB: **optimism under uncertainty**.


## Thompson Sampling Intuition

Thompson Sampling takes a Bayesian view. Instead of storing just one number for each arm, it stores a **distribution of plausible values** for how good that arm might be.

At each step:

1. sample one possible value from each arm's current belief distribution
2. choose the arm with the largest sampled value
3. update that arm's belief using the observed reward

So UCB adds an explicit uncertainty bonus, while Thompson Sampling handles uncertainty by random sampling from beliefs.


### Thompson Sampling Math for Bernoulli Rewards

Suppose each arm gives reward `1` for success and `0` for failure. Then a natural Bayesian model is:

- unknown success rate of an arm: `p`
- prior belief about `p`: `Beta(alpha, beta)`

If we observe:

- `s` successes
- `f` failures

then the posterior becomes:

`Beta(alpha + s, beta + f)`

A common beginner choice is the uniform prior `Beta(1, 1)`.

Intuition:

- more successes push the distribution to the right
- more failures push it to the left
- little data means a wide, uncertain distribution


## Worked Thompson Sampling Example

Suppose we start with a prior `Beta(1, 1)` for every arm.

Now imagine the observed outcomes are:

- arm A: 4 successes, 1 failure
- arm B: 1 success, 1 failure
- arm C: 1 success, 3 failures

Then the posteriors become:

- arm A: `Beta(5, 2)`
- arm B: `Beta(2, 2)`
- arm C: `Beta(2, 4)`

Arm A looks best on average, but arm B is still much more uncertain, so Thompson Sampling may still sample a high value for B sometimes.


In [3]:
posterior_params = {
    'A': {'alpha': 5, 'beta': 2},
    'B': {'alpha': 2, 'beta': 2},
    'C': {'alpha': 2, 'beta': 4},
}

posterior_summary = {}
for arm, params in posterior_params.items():
    alpha = params['alpha']
    beta = params['beta']
    posterior_mean = alpha / (alpha + beta)
    sampled_value = random.betavariate(alpha, beta)
    posterior_summary[arm] = {
        'posterior_mean': round(posterior_mean, 4),
        'one_sample': round(sampled_value, 4),
    }

posterior_summary


{'A': {'posterior_mean': 0.7143, 'one_sample': 0.6973},
 'B': {'posterior_mean': 0.5, 'one_sample': 0.2379},
 'C': {'posterior_mean': 0.3333, 'one_sample': 0.2443}}

Notice the difference between the posterior mean and the random sample. Thompson Sampling does not always choose the arm with the largest mean. It chooses the arm with the largest sampled draw from its current belief.

That randomness is not a bug. It is the mechanism that naturally balances exploration and exploitation.


## Simulate Both Methods on the Same Toy Bandit

Now let us use a three-arm Bernoulli bandit with true success probabilities:

- arm A: `0.30`
- arm B: `0.50`
- arm C: `0.70`

The learner does not know these values. It only sees successes and failures.


In [4]:
true_rates = {'A': 0.30, 'B': 0.50, 'C': 0.70}
arms = list(true_rates)

def pull_arm(arm):
    return 1 if random.random() < true_rates[arm] else 0

def run_ucb(steps=80):
    counts = {arm: 0 for arm in arms}
    rewards = {arm: 0 for arm in arms}

    # Pull each arm once so the UCB formula is well-defined.
    for arm in arms:
        reward = pull_arm(arm)
        counts[arm] += 1
        rewards[arm] += reward

    for t in range(len(arms) + 1, steps + 1):
        scores = {}
        for arm in arms:
            mean_reward = rewards[arm] / counts[arm]
            bonus = math.sqrt((2 * math.log(t)) / counts[arm])
            scores[arm] = mean_reward + bonus
        chosen = max(scores, key=scores.get)
        reward = pull_arm(chosen)
        counts[chosen] += 1
        rewards[chosen] += reward

    avg_rewards = {arm: round(rewards[arm] / counts[arm], 3) for arm in arms}
    return counts, rewards, avg_rewards

def run_thompson(steps=80):
    alpha = {arm: 1 for arm in arms}
    beta = {arm: 1 for arm in arms}
    counts = {arm: 0 for arm in arms}
    rewards = {arm: 0 for arm in arms}

    for _ in range(steps):
        samples = {arm: random.betavariate(alpha[arm], beta[arm]) for arm in arms}
        chosen = max(samples, key=samples.get)
        reward = pull_arm(chosen)

        counts[chosen] += 1
        rewards[chosen] += reward
        alpha[chosen] += reward
        beta[chosen] += 1 - reward

    avg_rewards = {arm: round(rewards[arm] / counts[arm], 3) if counts[arm] else 0.0 for arm in arms}
    posteriors = {arm: (alpha[arm], beta[arm]) for arm in arms}
    return counts, rewards, avg_rewards, posteriors

ucb_counts, ucb_rewards, ucb_avgs = run_ucb()
th_counts, th_rewards, th_avgs, th_posteriors = run_thompson()

{
    'ucb_counts': ucb_counts,
    'ucb_average_rewards': ucb_avgs,
    'thompson_counts': th_counts,
    'thompson_average_rewards': th_avgs,
    'thompson_posteriors': th_posteriors,
}


{'ucb_counts': {'A': 11, 'B': 27, 'C': 42},
 'ucb_average_rewards': {'A': 0.273, 'B': 0.593, 'C': 0.714},
 'thompson_counts': {'A': 4, 'B': 7, 'C': 69},
 'thompson_average_rewards': {'A': 0.25, 'B': 0.429, 'C': 0.667},
 'thompson_posteriors': {'A': (2, 4), 'B': (4, 5), 'C': (47, 24)}}

Both methods should usually spend more time on arm C because it is truly the best arm. But they get there in different ways:

- **UCB** is deterministic once the data is fixed. It adds a bonus term and chooses the largest score.
- **Thompson Sampling** is randomized. It samples from belief distributions and chooses the arm with the best draw.


## A Simple Comparison Summary

UCB:

- easy to explain as optimism under uncertainty
- uses an explicit exploration bonus
- often analyzed with clean regret bounds

Thompson Sampling:

- easy to explain as sampling from current beliefs
- naturally balances exploration and exploitation
- often works very well in practice


## Exercises and Pitfalls

Try these:

1. Change the true rates to make the arms closer together, like `0.45`, `0.50`, and `0.55`. Which method seems to need more exploration?
2. Increase the number of steps from `80` to `500`. Do both methods spend more of their pulls on the best arm?
3. For UCB, inspect what happens if an arm has a high mean but very few samples.
4. For Thompson Sampling, start from a stronger prior like `Beta(5, 5)`. How does that change the early behavior?

Common pitfalls:

- Thinking UCB's bonus is arbitrary. It is there to reflect uncertainty.
- Thinking Thompson Sampling only uses randomness for luck. The randomness is how it explores according to its beliefs.
- Forgetting that in small samples, observed averages can be very misleading.


In [ ]:
# Exercise scaffold: try a longer run for both methods.
random.seed(11)
run_ucb(steps=300), run_thompson(steps=300)[0]


# Contextual Bandit
> From ChatGPT

In [ ]:
import numpy as np
import random
from collections import defaultdict
from sklearn.linear_model import SGDRegressor

np.random.seed(42)
random.seed(42)

# -----------------------------
# Step 1: Simulated Environment
# This is your feature vector (x)
# -----------------------------
def generate_user():
    """
    Context: single feature (avg deposit)
    """
    return np.array([np.random.uniform(50, 2000)])


def get_reward(context, action):
    """
    True reward function (unknown to model)
    
    Let's define:
    - Action A works better for low-value users
    - Action B works better for high-value users
    """
    x = context[0]
    
    if action == 'A':
        base = 0.05 * x + 20   # better for low x
    else:
        base = 0.08 * x        # better for high x
    
    noise = np.random.normal(0, 20)
    return base + noise


# -----------------------------
# Step 2: Contextual Bandit
# -----------------------------
class ContextualEpsilonGreedy:
    def __init__(self, actions, epsilon=0.2):
        self.actions = actions
        self.epsilon = epsilon
        
        # One model per action
        self.models = {
            a: SGDRegressor(learning_rate='constant', eta0=0.001)
            for a in actions
        }
        
        self.is_fitted = {a: False for a in actions}
    
    def predict(self, context):
        """
        Predict expected reward for each action
        """
        preds = {}
        for a in self.actions:
            if self.is_fitted[a]:
                preds[a] = self.models[a].predict([context])[0]
            else:
                preds[a] = 0.0  # cold start
        return preds
    
    def select_action(self, context):
        """
        Epsilon-greedy decision
        """
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        
        preds = self.predict(context)
        return max(preds, key=preds.get)
    
    def update(self, context, action, reward):
        """
        Train only the model corresponding to chosen action
        """
        model = self.models[action]
        
        if not self.is_fitted[action]:
            model.partial_fit([context], [reward])
            self.is_fitted[action] = True
        else:
            model.partial_fit([context], [reward])


# -----------------------------
# Step 3: Run Simulation
# -----------------------------
def run_simulation(steps=500):
    bandit = ContextualEpsilonGreedy(actions=['A', 'B'], epsilon=0.2)
    
    history = []
    
    for step in range(steps):
        context = generate_user()
        
        action = bandit.select_action(context)
        reward = get_reward(context, action)
        
        bandit.update(context, action, reward)
        
        history.append((context[0], action, reward))
    
    return history


# -----------------------------
# Step 4: Execute
# -----------------------------
history = run_simulation(steps=500)

# Preview
history[:10]

[(np.float64(780.3532317523568), 'A', np.float64(36.780059226679434)),
 (np.float64(163.26304372798893), 'B', np.float64(19.439087192026783)),
 (np.float64(1222.1742728992572), 'A', np.float64(69.49115096449256)),
 (np.float64(643.2723737710986), 'A', np.float64(41.66022254499198)),
 (np.float64(892.2927863521257), 'A', np.float64(80.94354093663284)),
 (np.float64(619.6820646436754), 'B', np.float64(19.09704521917682)),
 (np.float64(939.3364692232201), 'B', np.float64(62.55741831937259)),
 (np.float64(1205.2084092809828), 'A', np.float64(92.2148298023013)),
 (np.float64(1234.7124612078048), 'A', np.float64(69.73054605827201)),
 (np.float64(1384.254401698706), 'B', np.float64(129.68914855682925))]